# Day 4 · Afternoon exercises (Exercise B)

**13:30–15:00 · then everyone back in the room at 15:00 for Peer review B**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. Keep the folders `data/` and `models/` next to this notebook.

**If Stan does not compile on your laptop**, continue on UCloud or on a group member's laptop.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| B1 | Three pooling models | 20 min |
| B2 | Shrinkage | 20 min |
| B3 | A person we know versus a new person | 15 min |
| B4 | **Challenge, no template:** does feedback make the class more alike? | 25 min |
| — | Exit response and your group's question set | 10 min |

**Roles this block:** rotate them from the morning.

**The data.** The Day 1 time game without feedback: each person tried ten times to wait **10 seconds** with eyes closed (we use only the 10-second target today). If the lecturer has placed the class's data as `data/timing-class.csv`, the notebook uses it; otherwise it uses `data/timing-backup.csv`, **synthetic** data for 15 fictional people. The column `elapsed_seconds` is the recorded time. This afternoon we use **all** people.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")
DATA <- "data"; MODELS <- "models"     # folders next to this notebook

In [ ]:
# The class's Day 1 data if the lecturer has put them here, otherwise the synthetic backup
file <- if (file.exists(file.path(DATA, "timing-class.csv"))) "timing-class.csv" else "timing-backup.csv"
file
d <- read_csv(file.path(DATA, file), show_col_types = FALSE) |>
  filter(target_seconds == 10, feedback_condition == "no_feedback", !is.na(elapsed_seconds))
d |> count(participant)

# Stan needs each person as a number 1, ..., J
people <- sort(unique(d$participant))
d <- d |> mutate(person = match(participant, people))
stan_data <- list(N = nrow(d), J = length(people), person = d$person, y = d$elapsed_seconds)
length(people)

## B1 (20 min): three pooling models

**The models** (*slides "Attempts are grouped within people", "Three ways to use the class" and "The hierarchical model"*). yᵢⱼ is attempt i of person j.

| model | file | person averages |
|---|---|---|
| complete pooling | `models/complete_pooling.stan` | one μ for everybody |
| no pooling | `models/no_pooling.stan` | θⱼ ~ Normal(10, 2), each on its own |
| partial pooling | `models/hierarchical.stan` | θⱼ ~ Normal(μ, τ), with μ and τ learned |

In all three, yᵢⱼ ~ Normal(person average, σ) with the same σ for everybody. `hierarchical.stan` is the non-centred version (*slide "Non-centred: same model, easier shape"*); it also generates predictions for B3.

**Tasks.**

1. Open the three `.stan` files. For each, say in one line what is shared between people and what is not.
2. Fit all three models with `stan_data` (4 chains, `seed = 2026`). Check the diagnostics of each before you compare anything. If a check fails, say what you would do and do it (*Day 3 slide "Six numbers to check after every fit"*).
3. From the partial-pooling fit, report μ, τ and σ with 90% intervals. In words: what do τ and σ describe here, and which is larger?

In [ ]:
# fit_complete <- cmdstan_model(file.path(MODELS, "complete_pooling.stan"))$sample(
#   data = stan_data, chains = 4, seed = 2026, refresh = 0)
# fit_nopool   <- ... "no_pooling.stan" ...
# fit_partial  <- ... "hierarchical.stan" ...

# diagnostics, for each fit:
# fit_partial$diagnostic_summary()
# max(fit_partial$summary()$rhat, na.rm = TRUE);  min(fit_partial$summary()$ess_bulk, na.rm = TRUE)

# fit_partial$summary(c("mu", "tau", "sigma"), "mean", ~quantile(.x, c(0.05, 0.95)))

**Your answer:** write a few sentences here.

## B2 (20 min): shrinkage

**The idea** (*slides "Where the weight w comes from", "Shrinkage: a compromise, again" and "Shrinkage in the class"*). With partial pooling each person's estimate is close to wⱼ · ȳⱼ + (1 − wⱼ) · μ, where ȳⱼ is her own average and

wⱼ = (nⱼ / σ²) / (nⱼ / σ² + 1 / τ²).

**Tasks.**

1. Make a table with, for each person: number of attempts nⱼ, own average ȳⱼ, the no-pooling estimate and the partial-pooling estimate of θⱼ (posterior means). Plot it as on the slide: an arrow from ȳⱼ to the partial-pooling estimate, and a line at μ.
2. Who moves most, and why? Compute wⱼ by hand for one person, using the posterior means of σ, τ and μ, and check the estimate against the table.
3. Now pretend each person made only **2 attempts**: refit the partial-pooling model with `d |> filter(trial <= 2)`. Make the same plot. What changes in the shrinkage, and why? (Use the formula for w.)

In [ ]:
# Task 1
# own <- d |> group_by(person, participant) |> summarise(n = n(), own_mean = mean(elapsed_seconds), .groups = "drop")
# own <- own |> mutate(no_pool = fit_nopool$summary("theta")$mean,
#                      partial = fit_partial$summary("theta")$mean)
# mu_hat <- fit_partial$summary("mu")$mean
# ggplot(own, aes(y = participant)) +
#   geom_vline(xintercept = mu_hat, colour = "orange") +
#   geom_segment(aes(x = own_mean, xend = partial, yend = participant), arrow = arrow(length = unit(0.1, "inches"))) +
#   geom_point(aes(x = own_mean, colour = "own average"), size = 3) +
#   geom_point(aes(x = partial, colour = "partial pooling"), size = 3) +
#   scale_colour_manual(values = c("own average" = "red", "partial pooling" = "navy"), name = NULL) +
#   labs(caption = "orange line: mu, the average of all people")

# Task 3
# d2 <- d |> filter(trial <= 2)
# stan_data2 <- list(N = nrow(d2), J = length(people), person = d2$person, y = d2$elapsed_seconds)

**Your answer:** write a few sentences here.

## B3 (15 min): a person we know versus a new person

**The idea** (*slide "Predicting: a person we know, or a new person"*). `hierarchical.stan` generates, in every draw:

- `y_next[j]`: the next attempt of person j in the data, ỹ ~ Normal(θⱼ, σ);
- `y_new_person`: an attempt by a **new** person, in two steps: θ_new ~ Normal(μ, τ), then ỹ ~ Normal(θ_new, σ).

**Tasks.**

1. Give the 90% interval for the next attempt of the person with the **highest** own average, of the person with the **lowest**, and of a new person.
2. Where is each interval centred, and why? Which is widest, and why?
3. Complete pooling predicts *every* attempt, by anybody, as ỹ ~ Normal(μ, σ) with its own μ and σ. Compute its interval for a new person and compare it with the partial-pooling one. Compare also the 90% intervals for μ in the two models (from B1). What does complete pooling get wrong?
4. Which assumption must hold for the new-person prediction to make sense?

In [ ]:
# draws_p <- fit_partial$draws(format = "df")
# highest and lowest own average (from B2):
# j_high <- own$person[which.max(own$own_mean)];  j_low <- own$person[which.min(own$own_mean)]
# quantile(draws_p[[paste0("y_next[", j_high, "]")]], c(0.05, 0.95))
# quantile(draws_p$y_new_person, c(0.05, 0.95))

**Your answer:** write a few sentences here.

*Optional:* fit the **centred** version `models/hierarchical_centered.stan` to the two-attempt data `stan_data2` and compare the number of divergences with the non-centred fit from B2 (*slides "A warning sign: the funnel" and "Non-centred: same model, easier shape"*).

## B4 (25 min): challenge, no template — does feedback make the class more alike?

*Slides "The hierarchical model", "The hierarchical model in Stan", "Non-centred: same model, easier shape" and "Predicting: a person we know, or a new person".* Use **all** 10-second attempts of the class in **both** games. There is no starter code: plan the steps in your group, then write them.

**Tasks.**

1. Write **one** hierarchical Stan program for both games: person j has an average θⱼ in each game; in game c the averages come from Normal(μ_c, τ_c), and single attempts scatter with σ_c. Use the non-centred form. Fit it and check the diagnostics.
2. Report μ, τ and σ for each game with 90% intervals, and P(τ_feedback < τ_without | y) and P(σ_feedback < σ_without | y). What does feedback do to the differences **between** people, and to the scatter **within** a person?
3. For each person, compute P(|θ_feedback − 10| < |θ_without − 10| | y): is her average closer to 10 s with feedback? How many of the 56 people have this probability above 0.9?
4. Give the 90% interval of a new person's next attempt in each game.

<details><summary><b>Hints</b> (open only when you are stuck)</summary>

- **Data:** read the file again without the `feedback_condition` filter, keep `target_seconds == 10`, and add `cond` (1 without, 2 with feedback) and `person` (as in B1, with `match()`).
- **Parameters:** `vector[2] mu; vector<lower=0>[2] tau; vector<lower=0>[2] sigma; matrix[J, 2] z;` — one column of z per game.
- **Person averages:** in `transformed parameters`, `matrix[J, 2] theta;` and `for (c in 1:2) theta[, c] = mu[c] + tau[c] * z[, c];`. `theta[, c]` is column c.
- **Priors:** as in `models/hierarchical.stan`; for the matrix, `to_vector(z) ~ std_normal();`.
- **Likelihood:** a loop is simplest: `for (i in 1:N) y[i] ~ normal(theta[person[i], cond[i]], sigma[cond[i]]);`.
- **A new person:** `normal_rng(normal_rng(mu[c], tau[c]), sigma[c])` for each game c.
- **Per person:** the draws have columns `theta[j,1]` and `theta[j,2]`; `paste0("theta[", j, ",1]")` builds the name. A `for` loop over j with `bind_rows()` collects the results.
- **Diagnostics:** if a bulk ESS is below 400, run longer with `iter_sampling = 2000`.

</details>

In [ ]:
# B4: your code

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: What does partial pooling share between people, and why do people with few attempts move most towards the group average? Why is a prediction for a new person wider than for a person in the data?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 15:00 for Peer review B.** Save this notebook (Ctrl/Cmd + S). At 15:45: daily diary on Mentimeter.